# EMA 6938 — Data Science for Materials
## Week 4 Notebook: Exploratory Data Analysis

**Name:** *(your name here)*  
**Date:** *(date)*  
**Kernel:** Python (matds)

---

**Chapters:** Sandfeld Ch. 9–10  
**Format:** Take-home. Due **Sunday 11:59 PM**  
**Dataset:** `data/week4_mp_oxides.csv` (instructor-provided, save in the same folder as this notebook)

This notebook has six parts:

| Part | Title | Connects to |
|------|-------|-------------|
| A | Load & Inspect | Lecture Segment 2 |
| B | Univariate Analysis | Lecture Segment 3 |
| C | Bivariate & Multivariate Analysis | Lecture Segment 4 |
| D | Stratified Analysis | Lecture Segment 3 |
| E | Composition Featurization | Lecture Segment 5 |
| F | Reflection | All segments |

Parts A–D can be started during the in-class lab session. Parts E–F are take-home.

**Submission:** Upload this `.ipynb` file to Canvas. Run `Kernel → Restart & Run All` before submitting to confirm all cells execute cleanly.

> **AI tool disclosure:** If you used any AI coding assistant (GitHub Copilot, ChatGPT, etc.) while completing this notebook, describe briefly which tool, for what purpose, and what you verified yourself. Delete this line if no AI tools were used.

In [ ]:
# Cell A0 — Environment check
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore, iqr
import setuptools
import matminer
import pkg_resources

print(f"Python:     {sys.version.split()[0]}")
print(f"NumPy:      {np.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"seaborn:    {sns.__version__}")
print(f"matminer:   {matminer.__version__}")
print(f"setuptools: {setuptools.__version__}")
print(f"pkg_resources: {pkg_resources.__file__}")
print("\n✅ Environment OK — ready to begin")

---
## Part A — Load & Inspect
**Connects to: Lecture Segment 2, Sandfeld Ch. 9**

Before any analysis, always know your data: size, types, and missingness.
Every number from `df.describe()` should be checked for physical plausibility before you plot anything.

### A1 — Load the dataset and run the initial inspection
**Lecture demo — reproduce and understand**

In [ ]:
# Cell A1 — Load and inspect the MP oxide dataset
# LECTURE DEMO

df = pd.read_csv('data/week4_mp_oxides.csv')

print(f"Shape:   {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nStatistical summary:")
df.describe().round(3)

### A2 — Crystal systems and metallic fraction
**Lecture demo**

In [ ]:
# Cell A2 — Crystal system counts and metallic fraction
# LECTURE DEMO

print("Crystal system value counts:")
print(df['crystal_system'].value_counts())
print()

n_metallic = (df['band_gap'] == 0).sum()
frac_metallic = n_metallic / len(df)
print(f"Metallic entries (band_gap = 0): {n_metallic:,}  ({frac_metallic:.1%} of dataset)")
print(f"Non-metallic (band_gap > 0):     {(df['band_gap'] > 0).sum():,}")

### A3 — Task: investigate missing values

In [ ]:
# Cell A3 — Task: investigate missing values
# YOUR CODE HERE

# 1. Identify which column has the most missing values
# 2. Find 3 entries where that column is missing, print their mp_id, formula,
#    and band_gap so you can reason about why the value might be absent
# 3. In the reflection cell below, explain what you would do before
#    using this dataset to train a model

missing = df.isnull().sum().sort_values(ascending=False)
print("Missing values per column (top 5):")
print(missing[missing > 0].head())
most_missing_col = missing.idxmax()
print(f"\nColumn with most missing values: '{most_missing_col}' ({missing[most_missing_col]} missing)")

# 2. Find 3 entries where that column is missing
missing_rows = df[df[most_missing_col].isnull()][['mp_id','formula','band_gap']].head(3)
print(f"\n3 entries where '{most_missing_col}' is missing:")
print(missing_rows.to_string(index=False))


**A3 Reflection** *(answer in this cell)*

In 1–2 sentences: why might that property be missing for those particular entries?
What would you do before using this dataset to train a model?

*Your answer here:*


---
## Part B — Univariate Analysis
**Connects to: Lecture Segment 3, Sandfeld Ch. 9**

Examine each property individually: distribution shape, central tendency, spread, and outliers.
The question for every property: *What does the distribution look like? Where is it centred? How spread is it? Are there outliers?*

### B1 — Band gap: histogram + KDE overlay
**Lecture demo — reproduce and understand**

In [ ]:
# Cell B1 — Band gap distribution: full dataset + filtered non-metals
# LECTURE DEMO

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Full distribution (metals + insulators)
axes[0].hist(df['band_gap'].dropna(), bins=60, color='#1C2B4A', alpha=0.85,
             density=True, edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Band gap (eV)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title(f'Full dataset  (n = {df["band_gap"].notna().sum():,})', fontsize=11)

# Non-metals only (band_gap > 0.1 eV)
bg_nm = df[df['band_gap'] > 0.1]['band_gap'].dropna()
axes[1].hist(bg_nm, bins=60, color='#0D9488', alpha=0.85,
             density=True, edgecolor='white', linewidth=0.4)
from scipy.stats import gaussian_kde
x_kde = np.linspace(bg_nm.min(), bg_nm.max(), 300)
axes[1].plot(x_kde, gaussian_kde(bg_nm)(x_kde), color='#1C2B4A', lw=2, label='KDE')
axes[1].set_xlabel('Band gap (eV)', fontsize=11)
axes[1].set_title(f'Non-metals only  (n = {len(bg_nm):,})', fontsize=11)
axes[1].legend()

plt.suptitle('Band gap distribution — MP oxide dataset', fontsize=12, y=1.02)
plt.tight_layout()
#plt.savefig('B1_bandgap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

**B1 Reflection** *(answer in this cell)*

In 2–3 sentences: describe the shape of the full band gap distribution.
Is it unimodal or bimodal? Symmetric or skewed?
What physical phenomenon explains the spike at 0 eV?

*Your answer here:*


### B2 — Formation energy: histogram with mean and median
**Lecture demo**

In [ ]:
# Cell B2 — Formation energy distribution with mean/median overlay
# LECTURE DEMO

fig, ax = plt.subplots(figsize=(9, 4))

ef_data = df['Ef_eV_atom'].dropna()
ax.hist(ef_data, bins=60, color='#7C3AED', alpha=0.80,
        density=True, edgecolor='white', linewidth=0.4)

mean_ef   = ef_data.mean()
median_ef = ef_data.median()
ax.axvline(mean_ef,   color='#EF4444', lw=1.8, ls='--',  label=f'Mean   {mean_ef:.3f} eV/atom')
ax.axvline(median_ef, color='#F59E0B', lw=1.8, ls=':',   label=f'Median {median_ef:.3f} eV/atom')

ax.set_xlabel('Formation energy (eV/atom)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title(f'Formation energy distribution  (n = {len(ef_data):,})', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
#plt.savefig('B2_formation_energy_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean:   {mean_ef:.4f} eV/atom")
print(f"Median: {median_ef:.4f} eV/atom")
print(f"Difference (mean − median): {mean_ef - median_ef:.4f} eV/atom")

### B3 — IQR outlier detection on band gap
**Lecture demo**

In [ ]:
# Cell B3 — IQR outlier detection: band gap
# LECTURE DEMO
# Compute IQR on non-metals only — including band_gap = 0 (metals)
# collapses Q1 to 0 and produces almost no upper outliers
bg_nonmetal = df[df['band_gap'] > 0.1]['band_gap']

Q1  = bg_nonmetal.quantile(0.25)
Q3  = bg_nonmetal.quantile(0.75)
IQR_val = Q3 - Q1
lo  = Q1 - 1.5 * IQR_val
hi  = Q3 + 1.5 * IQR_val

outliers = df[(df['band_gap'] > 0.1) &
              ((df['band_gap'] < lo) | (df['band_gap'] > hi))]

print(f"Q1 = {Q1:.3f} eV   Q3 = {Q3:.3f} eV   IQR = {IQR_val:.3f} eV")
print(f"IQR fences: lower = {lo:.3f} eV   upper = {hi:.3f} eV")
print(f"Flagged as outliers: {len(outliers):,}  ({100*len(outliers)/len(df):.1f}% of dataset)")
print()
print("10 largest band gap outliers:")
print(outliers.nlargest(10, 'band_gap')[['mp_id','formula','band_gap','crystal_system']].to_string(index=False))

**B3 Reflection** *(answer in this cell)*

Pick two flagged outliers. For each: (1) name the material, (2) state its band gap,
(3) explain in one sentence whether you would keep or remove it and why.

*Your answer here:*


### B4 — Task: z-score outlier detection on density

In [ ]:
# Cell B4 — Task: z-score outlier detection on density_g_cm3
# YOUR CODE HERE

# 1. Compute z-scores for density_g_cm3 using scipy.stats.zscore
#    (drop NaNs first — handle them explicitly, do not pass NaN to zscore)
# 2. Flag entries with |z| > 3 as outliers
# 3. Print: number flagged, percentage, and the top 5 by |z-score|
#    with columns: mp_id, formula, density_g_cm3, z_score
# 4. Compare to IQR: does z-score flag more or fewer entries than IQR for density?

from scipy.stats import zscore

density_data = df['density_g_cm3'].dropna()
z_scores = zscore(density_data)

df_density = df[df['density_g_cm3'].notna()].copy()
df_density['z_score'] = z_scores

outliers_z = df_density[df_density['z_score'].abs() > 3]
print(f"Z-score outliers (|z| > 3): {len(outliers_z):,}  ({100*len(outliers_z)/len(density_data):.1f}%)")
print()
print("Top 5 density outliers by |z-score|:")
print(outliers_z.nlargest(5, 'z_score')[['mp_id','formula','density_g_cm3','z_score']].to_string(index=False))

# Compare to IQR for density
Q1_d = density_data.quantile(0.25)
Q3_d = density_data.quantile(0.75)
IQR_d = Q3_d - Q1_d
outliers_iqr = df_density[(df_density['density_g_cm3'] < Q1_d - 1.5*IQR_d) |
                           (df_density['density_g_cm3'] > Q3_d + 1.5*IQR_d)]
print(f"\nComparison — density outliers:")
print(f"  IQR method (1.5×IQR):  {len(outliers_iqr):,}")
print(f"  Z-score method (|z|>3): {len(outliers_z):,}")
print(f"  IQR flags {'more' if len(outliers_iqr) > len(outliers_z) else 'fewer'} entries than z-score")


### B5 — Box plots: three properties side by side
**Lecture demo**

In [ ]:
# Cell B5 — Box plots for band_gap, Ef_eV_atom, density_g_cm3
# LECTURE DEMO

props  = ['band_gap', 'Ef_eV_atom', 'density_g_cm3']
labels = ['Band gap (eV)', 'Formation energy (eV/atom)', 'Density (g/cm³)']
colors = ['#1C2B4A', '#7C3AED', '#0D9488']

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

for ax, prop, label, color in zip(axes, props, labels, colors):
    data = df[prop].dropna()
    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='white', linewidth=2),
               flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax.set_ylabel(label, fontsize=10)
    ax.set_title(f'{label}', fontsize=10)
    ax.set_xticks([])

plt.suptitle('Box plots — MP oxide dataset', fontsize=12)
plt.tight_layout()
#plt.savefig('B5_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part C — Bivariate & Multivariate Analysis
**Connects to: Lecture Segment 4, Sandfeld Ch. 9**

Explore relationships between pairs and groups of properties.
Use both Pearson and Spearman — materials data is rarely linear and rarely Gaussian.

### C1 — Scatter: formation energy vs. band gap
**Lecture demo**

In [ ]:
# Cell C1 — Scatter: Ef_eV_atom vs. band_gap, coloured by crystal_system
# LECTURE DEMO

from scipy.stats import pearsonr, spearmanr

df_c1 = df[['band_gap','Ef_eV_atom','crystal_system']].dropna()
systems = df_c1['crystal_system'].unique()
cmap    = plt.cm.tab10(np.linspace(0, 0.9, len(systems)))

fig, ax = plt.subplots(figsize=(9, 6))
for sys, color in zip(systems, cmap):
    sub = df_c1[df_c1['crystal_system'] == sys]
    ax.scatter(sub['band_gap'], sub['Ef_eV_atom'],
               color=color, alpha=0.45, s=12, label=sys)

ax.set_xlabel('Band gap (eV)', fontsize=11)
ax.set_ylabel('Formation energy (eV/atom)', fontsize=11)
ax.set_title('Formation energy vs. Band gap — coloured by crystal system', fontsize=11)
ax.legend(fontsize=8, ncol=2, title='Crystal system')

r_p, p_p = pearsonr(df_c1['band_gap'], df_c1['Ef_eV_atom'])
r_s, p_s = spearmanr(df_c1['band_gap'], df_c1['Ef_eV_atom'])
ax.text(0.02, 0.97, f"Pearson r = {r_p:.3f}  (p = {p_p:.2e})\nSpearman ρ = {r_s:.3f}  (p = {p_s:.2e})",
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.tight_layout()
#plt.savefig('C1_scatter_ef_bg.png', dpi=150, bbox_inches='tight')
plt.show()

**C1 Reflection** *(answer in this cell)*

In 2–3 sentences: which correlation coefficient (Pearson or Spearman) is more appropriate here and why?
Is there evidence of a confounding variable? What physical mechanism could explain any correlation between
formation energy and band gap?

*Your answer here:*


### C2 — Spearman correlation heatmap
**Lecture demo**

In [ ]:
# Cell C2 — Spearman correlation heatmap
# LECTURE DEMO

cols = ['band_gap', 'Ef_eV_atom', 'density_g_cm3', 'volume_A3', 'nsites']
corr = df[cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Spearman correlation matrix — MP oxide dataset', fontsize=11)
plt.tight_layout()
#plt.savefig('C2_spearman_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Spearman correlation matrix:")
print(corr.round(3))

**C2 Reflection** *(answer in this cell)*

Identify the two strongest and two weakest correlations in the heatmap (excluding the diagonal).
For each of the two strongest, provide a physical explanation — why are these properties correlated?

*Your answer here:*


### C3 — Pairplot
**Lecture demo**

In [ ]:
# Cell C3 — Pairplot for three numeric properties by crystal system
# LECTURE DEMO
# Note: subsample first — pairplot is slow with >1,000 points

df_pp = df[['band_gap','Ef_eV_atom','density_g_cm3','crystal_system']].dropna()
df_sample = df_pp.sample(min(500, len(df_pp)), random_state=42)

g = sns.pairplot(df_sample, hue='crystal_system',
                 vars=['band_gap','Ef_eV_atom','density_g_cm3'],
                 plot_kws=dict(alpha=0.75, s=15), palette='tab10',
                 diag_kind='hist')
g.figure.suptitle('Pairplot — Band gap, Formation energy, Density (n=500 sample)',
                   y=1.02, fontsize=11)
plt.savefig('C3_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part D — Stratified Analysis
**Connects to: Lecture Segment 3, Sandfeld Ch. 10**

Group by crystal system and examine whether property distributions differ across groups.
If distributions differ, that structural variable carries predictive information for ML.

### D1 — Group statistics by crystal system
**Lecture demo**

In [ ]:
# Cell D1 — Group statistics: band_gap by crystal_system
# LECTURE DEMO

stats = (df.groupby('crystal_system')['band_gap']
           .agg(['count','mean','median','std'])
           .rename(columns={'count':'n','mean':'mean_eV',
                            'median':'median_eV','std':'std_eV'})
           .sort_values('mean_eV', ascending=False))

print("Band gap statistics by crystal system:")
print(stats.round(3).to_string())

### D2 — Violin plot: band gap by crystal system
**Lecture demo**

In [ ]:
# Cell D2 — Violin plot: band_gap by crystal_system (top 5 most common)
# LECTURE DEMO

top5   = df['crystal_system'].value_counts().index[:5]
df_top5 = df[df['crystal_system'].isin(top5)].copy()

fig, ax = plt.subplots(figsize=(11, 5))

sns.violinplot(data=df_top5, x='crystal_system', y='band_gap',
               order=top5, palette='tab10', legend=False,
               inner='box', ax=ax)

ax.set_xlabel('Crystal system', fontsize=11)
ax.set_ylabel('Band gap (eV)', fontsize=11)
ax.set_title('Band gap distribution by crystal system (top 5)', fontsize=11)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
#plt.savefig('D2_violin_bandgap.png', dpi=150, bbox_inches='tight')
plt.show()

**D2 Reflection** *(answer in this cell)*

In 2–3 sentences: which crystal system shows the widest within-group variance for band gap?
Propose a physical reason, what structural or compositional diversity within that crystal system
could explain the wide spread?

*Your answer here:*


### D3 — Task: repeat stratified analysis for formation energy

In [ ]:
# Cell D3 — Task: stratified analysis for Ef_eV_atom
# YOUR CODE HERE

# 1. Compute the same group statistics table as D1, but for Ef_eV_atom
#    (count, mean, median, std) — sorted by mean_eV descending

# 2. Produce a violin plot of Ef_eV_atom by crystal_system (top 5 systems)
#    Save as 'D3_violin_ef.png'

# 3. In the reflection cell below, compare the patterns:
#    Does crystal system separate Ef distributions as well as it separates band gap?




**D3 Reflection** *(answer in this cell)*

Does crystal system explain more variance in band gap or in formation energy?
What does this tell you about the relative importance of crystal symmetry for each property?

*Your answer here:*


---
## Part E — Composition Featurization
**Connects to: Lecture Segment 5, Sandfeld Ch. 10**

Use Matminer to extract MAGPIE composition descriptors from chemical formulas.
These 132 features will be the input to the Week 5 random forest. You are building that feature matrix now.

### E1 — Convert formula strings to Composition objects
**Lecture demo — reproduce and understand**

In [ ]:
# Cell E1 — StrToComposition: convert formula strings to pymatgen Composition objects
# LECTURE DEMO

from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty

# Work on a small subset (formula + targets only) for speed
df_feat = df[['mp_id','formula','band_gap','Ef_eV_atom']].dropna().copy()
print(f"Working with {len(df_feat):,} entries")

# Convert formula strings → pymatgen Composition objects
stc = StrToComposition(target_col_id='composition')
df_feat = stc.featurize_dataframe(df_feat, 'formula', ignore_errors=True)

print("\nFirst 3 Composition objects:")
for _, row in df_feat.head(3).iterrows():
    print(f"  {row['formula']:12s}  →  {row['composition']}")

### E2 — Apply the MAGPIE featurizer
**Lecture demo**

In [ ]:
# Cell E2 — ElementProperty (MAGPIE preset): 132 features per formula
# LECTURE DEMO

ep = ElementProperty.from_preset('magpie')
print(f"MAGPIE generates {len(ep.feature_labels())} features per formula")
print("\nFirst 10 feature labels:")
for label in ep.feature_labels()[:10]:
    print(f"  {label}")

# Featurize — ignore_errors=True skips entries that fail (unusual elements/oxidation states)
df_feat = ep.featurize_dataframe(df_feat, 'composition', ignore_errors=True)
print(f"\nFeaturized dataset shape: {df_feat.shape}")

### E3 — Scatter: mean electronegativity vs. band gap
**Lecture demo**

In [ ]:
# Cell E3 — Mean electronegativity vs. band_gap
# LECTURE DEMO

from scipy.stats import pearsonr

feat_en = 'MagpieData mean Electronegativity'

if feat_en in df_feat.columns:
    df_plot = df_feat[[feat_en, 'band_gap', 'Ef_eV_atom']].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, yprop, ylabel, color in zip(
            axes,
            ['band_gap', 'Ef_eV_atom'],
            ['Band gap (eV)', 'Formation energy (eV/atom)'],
            ['#1C2B4A', '#7C3AED']):
        ax.scatter(df_plot[feat_en], df_plot[yprop],
                   alpha=0.35, s=10, color=color)
        r, p = pearsonr(df_plot[feat_en], df_plot[yprop])
        ax.set_xlabel('Mean electronegativity', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(f'Mean EN vs. {ylabel.split(" (")[0]}', fontsize=11)
        ax.text(0.02, 0.97, f'Pearson r = {r:.3f}\np = {p:.2e}',
                transform=ax.transAxes, va='top', fontsize=9,
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))

    plt.suptitle('Mean Electronegativity — MAGPIE feature', fontsize=12)
    plt.tight_layout()
    plt.savefig('E3_mean_EN.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f"Column '{feat_en}' not found. Available columns: {[c for c in df_feat.columns if 'Electro' in c]}")

### E4 — Task: explore one MAGPIE feature of your choice

In [ ]:
# Cell E4 — Task: choose one MAGPIE feature and plot vs. band_gap
# YOUR CODE HERE

# 1. Print the full list of feature labels to browse available options:
#    print(ep.feature_labels())

# 2. Choose any one feature that you expect to be physically relevant to band_gap.
#    Avoid mean ElectroNegativity (already shown in E3).

# 3. Make a scatter plot of your chosen feature vs. band_gap.
#    Compute and annotate the Pearson r value.
#    Save as 'E4_magpie_feature.png'

# 4. Answer the reflection question below.


**E4 Reflection** *(answer in this cell)*

State the feature you chose and why you expected it to correlate with band_gap.
Was the correlation in the direction you predicted?
Provide a physical explanation for the relationship (or lack thereof).

*Your answer here:*


---
## Part F — Reflection
**Complete after finishing Parts A–E**

### F1 — Most important EDA finding

In 3–4 sentences: what is the single most important thing you learned about the
MP oxide dataset from this EDA that you would *not* have known if you had gone
straight to model training without any exploration?
Be specific. Refer to an actual finding from one of the cells above.

*Your answer here:*


### F2 — Connection to Week 3

Revisit your Week 3 discussion post, where you described a measurement from your own research.
Does the distribution of that property (or a proxy in the MP dataset) behave as you predicted?
If not, what does the actual distribution tell you that the predicted one missed?

*Your answer here:*


---
## Day 2 Live Demo — MAGPIE Correlation Heatmap

> **This section is covered during the Day 2 deep dive session.**
> It is not part of the graded take-home — run it along with the instructor
> to explore feature redundancy in the full MAGPIE descriptor set.

Day 2 Demo — Full MAGPIE Spearman correlation heatmap

Instructor-led demo — run this during the Day 2 session

In [ ]:
# Cell Day2-1 — Full 132×132 MAGPIE Spearman correlation heatmap
# LECTURE DEMO

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── 1. Identify MAGPIE feature columns ───────────────────────────────────
non_feature_cols = ['mp_id', 'formula', 'band_gap', 'Ef_eV_atom',
                    'density_g_cm3', 'volume_A3', 'nsites', 'n_elements',
                    'crystal_system', 'composition', 'is_theoretical']
magpie_cols = [c for c in df_feat.columns
               if c not in non_feature_cols
               and df_feat[c].dtype in ['float64', 'float32']]

print(f"MAGPIE feature columns: {len(magpie_cols)}")

# ── 2. Compute Spearman correlation on a sample for speed ─────────────────
df_sample = df_feat[magpie_cols].dropna().sample(
    min(2000, len(df_feat)), random_state=42
)
corr_full = df_sample.corr(method='spearman')

print(f"Correlation matrix shape: {corr_full.shape}")

In [ ]:
# Cell Day2-2 — Plot the full heatmap (overview — labels hidden at this scale)
# LECTURE DEMO

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_full, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False,
            ax=ax, square=True, linewidths=0)
ax.set_title(
    f"Spearman correlation — all {len(magpie_cols)} MAGPIE features\n"
    "(labels hidden — too dense to read at this scale)",
    fontsize=11
)
plt.tight_layout()
plt.savefig('Day2_magpie_full_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("What to look for:")
print("  Dark red blocks = groups of highly positively correlated features")
print("  Dark blue blocks = groups of highly negatively correlated features")
print("  Pale regions = near-zero correlation (independent features)")

In [ ]:
# Cell Day2-3 — Identify the most redundant feature pairs (|ρ| > 0.95)
# LECTURE DEMO

# Extract upper triangle (avoid duplicate pairs and self-correlations)
upper = corr_full.where(
    np.triu(np.ones(corr_full.shape, dtype=bool), k=1)
)

# Find pairs with |ρ| > 0.95
redundant = (upper.stack()
               .reset_index()
               .rename(columns={'level_0':'feature_1','level_1':'feature_2',0:'rho'}))
redundant['abs_rho'] = redundant['rho'].abs()
redundant = redundant[redundant['abs_rho'] > 0.95].sort_values('abs_rho', ascending=False)

print(f"Feature pairs with |ρ| > 0.95: {len(redundant)}")
print()
print("Top 15 most redundant pairs:")
print(redundant.head(15)[['feature_1','feature_2','rho']]
      .to_string(index=False))

In [ ]:
# Cell Day2-4 — Zoom in: heatmap for one redundant feature group
# LECTURE DEMO
# Show the block of electronegativity-related features up close

en_cols = [c for c in magpie_cols if 'Electronegativity' in c or 'NValence' in c
           or 'NsValence' in c or 'NpValence' in c]

print(f"EN/valence-related features: {len(en_cols)}")
print(en_cols)

if len(en_cols) >= 4:
    corr_en = df_sample[en_cols].corr(method='spearman')
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr_en, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, square=True,
                linewidths=0.5, ax=ax)
    ax.set_title('Spearman correlation — Electronegativity & Valence features', fontsize=11)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig('Day2_en_valence_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nDiscussion: which of these features would Lasso keep? Which would it zero out?")

Day 2 Discussion questions:

Look at the top redundant pairs in Cell Day2-3. Do the most correlated feature pairs make physical sense — are they statistics of the same underlying elemental property?

In the zoomed heatmap (Cell Day2-4): which features within the EN/valence group are most correlated? Which are most independent?

If you were building a linear regression model, which features from this redundant group would you keep? How does this connect to what Lasso does automatically in Week 6?

---
## Submission Checklist

Before submitting via Canvas (due **Sunday 11:59 PM**):

**Part A — Load & Inspect**
- [ ] A1: Dataset loaded; shape, dtypes, and missing values printed
- [ ] A2: Crystal system counts and metallic fraction printed
- [ ] A3: Task cell contains working code; reflection answered

**Part B — Univariate Analysis**
- [ ] B1: Histogram + KDE plot saved (`B1_bandgap_distribution.png`); reflection answered
- [ ] B2: Formation energy distribution with mean/median lines saved
- [ ] B3: IQR outlier table printed; reflection answered
- [ ] B4: Task cell contains working code (z-score outlier detection)
- [ ] B5: Box plot figure saved (`B5_boxplots.png`)

**Part C — Bivariate & Multivariate**
- [ ] C1: Scatter plot saved (`C1_scatter_ef_bg.png`); reflection answered
- [ ] C2: Spearman heatmap saved (`C2_spearman_heatmap.png`); reflection answered
- [ ] C3: Pairplot saved (`C3_pairplot.png`)

**Part D — Stratified Analysis**
- [ ] D1: Group statistics table printed
- [ ] D2: Violin plot saved (`D2_violin_bandgap.png`); reflection answered
- [ ] D3: Task cell contains working code; violin plot saved (`D3_violin_ef.png`); reflection answered

**Part E — Composition Featurization**
- [ ] E1: Composition objects created; first 3 printed
- [ ] E2: MAGPIE featurization complete; shape printed
- [ ] E3: Mean EN scatter plots saved (`E3_mean_EN.png`)
- [ ] E4: Task cell contains working code; scatter saved (`E4_magpie_feature.png`); reflection answered

**Part F — Reflection**
- [ ] F1: Written answer (3–4 sentences, specific to a finding above)
- [ ] F2: Written answer connecting to Week 3 discussion

- [ ] AI disclosure note updated or deleted at the top of the notebook
- [ ] File renamed: `[LastName]_week4.ipynb`
**Final check:** Run `Kernel → Restart & Run All`. All cells must execute without errors before submitting.